In [8]:
!pip install -U google-genai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.7/724.7 kB 1.1 MB/s  0:00:00 eta 0:00:01
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 2.5 MB/s  0:00:02m0:00:0100:01
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.42.1
    Uninstalling google-auth-2.42.1:
      Successfully uninstalled google-auth-2.42.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [google-genai] [google-genai]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Non-batch version: translate ~N QAs per ONE LLM call (chunking), then write train-sin.json
# - Reads:  ../data/5k_qas/test copy.json
# - Writes: ../data/5k_qas/train-sin.json
# - Writes failures (qa_ids) to: ../data/5k_qas/failed-qaids.txt
#
# pip install -U google-genai

import os
import json
import time
from pathlib import Path
from typing import Dict, Any, List, Tuple

from google import genai

# -----------------------------
# Config
# -----------------------------
API_KEY = ""  # or use env: os.environ["GEMINI_API_KEY"]
MODEL_NAME = "models/gemini-3-flash-preview"

file = "test"
IN_PATH  = Path(f"../visual-genome/40k/40k-qas-{file}.json")
OUT_PATH = Path(f"../visual-genome/40k/40k-qas-{file}-sin.json")
FAILED_QAIDS_PATH = Path(f"../visual-genome/40k/failed-qaids.txt")

CHUNK_QAS_PER_CALL = 5   # << this is what you asked
SLEEP_BETWEEN_CALLS = 0.2     # small pause to avoid rate spikes (tune as needed)

GEN_CONFIG = {
    "temperature": 0.2,
    "max_output_tokens": 10000,   # bigger because we return 10 translations at once
}

SYSTEM_INSTRUCTION = (
    "You are a Sinhala (සිංහල) translator.\n"
    "Translate English to casual spoken Sinhala.\n"
    "Return ONLY valid JSON. No markdown. No extra text."
)

def chunk_list(xs: List[Any], n: int):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def build_prompt_for_chunk(items: List[Dict[str, Any]]) -> str:
    """
    items is list of dicts with qa_id, question, answer
    We force output to be JSON array of objects keyed by qa_id.
    """
    lines = []
    for it in items:
        qa_id = it["qa_id"]
        q = it["question"]
        a = it["answer"]
        lines.append(
            f'- qa_id: {qa_id}\n'
            f'  question: "{q}"\n'
            f'  answer: "{a}"\n'
        )

    return (
        "Translate each (question, answer) to Sinhala.\n"
        "Return ONLY a JSON array exactly like:\n"
        '[{"qa_id":123,"question":"...","answer":"..."}, ...]\n'
        "Rules:\n"
        "- Keep qa_id unchanged (number)\n"
        "- Output must have the SAME number of items as input\n"
        "- Do not add any keys other than qa_id, question, answer\n\n"
        "INPUT:\n" + "\n".join(lines)
    )

def try_parse_json_array(text: str) -> List[Dict[str, Any]]:
    """
    Parse model output, salvage first [...] if extra junk appears.
    """
    text = (text or "").strip()
    if not text:
        return []

    # direct
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    # salvage first [...]
    l = text.find("[")
    r = text.rfind("]")
    if 0 <= l < r:
        candidate = text[l:r+1]
        try:
            obj = json.loads(candidate)
            if isinstance(obj, list):
                return obj
        except Exception:
            pass

    return []

def flatten_qas(data: List[Dict[str, Any]]) -> List[Tuple[int, int, Dict[str, Any]]]:
    """
    Returns list of (item_i, qa_i, qa_dict_reference)
    """
    flat = []
    for item_i, item in enumerate(data):
        for qa_i, qa in enumerate(item.get("qas", [])):
            flat.append((item_i, qa_i, qa))
    return flat

# -----------------------------
# Main
# -----------------------------
if not API_KEY:
    API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not API_KEY:
    raise RuntimeError("Set API_KEY or export GEMINI_API_KEY.")

client = genai.Client(api_key=API_KEY)

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
out_data = json.loads(json.dumps(data, ensure_ascii=False))  # deep copy

flat = flatten_qas(out_data)
total = len(flat)
if total == 0:
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUT_PATH.write_text(json.dumps(out_data, ensure_ascii=False, indent=2), encoding="utf-8")
    raise SystemExit("No QAs found.")

failed_qaids: List[int] = []
done = 0

# We build a simpler list for prompting
qa_rows = []
for item_i, qa_i, qa in flat:
    qa_rows.append({
        "item_i": item_i,
        "qa_i": qa_i,
        "qa_id": qa["qa_id"],
        "question": qa["question"],
        "answer": qa["answer"],
    })

num_calls = (len(qa_rows) + CHUNK_QAS_PER_CALL - 1) // CHUNK_QAS_PER_CALL

for call_idx, chunk in enumerate(chunk_list(qa_rows, CHUNK_QAS_PER_CALL), start=1):
    print(f"[{call_idx}/{num_calls}] {min(call_idx*CHUNK_QAS_PER_CALL, total)}/{total}")

    prompt = build_prompt_for_chunk(chunk)

    try:
        resp = client.models.generate_content(
            model=MODEL_NAME,
            contents=[{"role": "user", "parts": [{"text": prompt}]}],
            config={
                "system_instruction": SYSTEM_INSTRUCTION,
                "temperature": GEN_CONFIG["temperature"],
                "max_output_tokens": GEN_CONFIG["max_output_tokens"],
            }
        )
        out_text = getattr(resp, "text", "") or ""
    except Exception:
        # whole chunk failed
        for row in chunk:
            failed_qaids.append(int(row["qa_id"]))
        continue

    parsed = try_parse_json_array(out_text)
    # Build map qa_id -> (q_si, a_si)
    trans_map: Dict[int, Dict[str, str]] = {}
    for obj in parsed:
        try:
            qa_id = int(obj["qa_id"])
            q_si = str(obj["question"])
            a_si = str(obj["answer"])
            trans_map[qa_id] = {"question": q_si, "answer": a_si}
        except Exception:
            continue

    # Write back, mark missing as failed
    for row in chunk:
        qa_id = int(row["qa_id"])
        item_i = row["item_i"]
        qa_i = row["qa_i"]
        if qa_id in trans_map:
            out_data[item_i]["qas"][qa_i]["question"] = trans_map[qa_id]["question"]
            out_data[item_i]["qas"][qa_i]["answer"] = trans_map[qa_id]["answer"]
            done += 1
        else:
            failed_qaids.append(qa_id)

    # Persist progress each call (safe for long runs)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUT_PATH.write_text(json.dumps(out_data, ensure_ascii=False, indent=2), encoding="utf-8")

    FAILED_QAIDS_PATH.parent.mkdir(parents=True, exist_ok=True)
    FAILED_QAIDS_PATH.write_text("\n".join(map(str, sorted(set(failed_qaids)))) + ("\n" if failed_qaids else ""), encoding="utf-8")

    time.sleep(SLEEP_BETWEEN_CALLS)

# Final save
OUT_PATH.write_text(json.dumps(out_data, ensure_ascii=False, indent=2), encoding="utf-8")
FAILED_QAIDS_PATH.write_text("\n".join(map(str, sorted(set(failed_qaids)))) + ("\n" if failed_qaids else ""), encoding="utf-8")

print(f"Done. Translated: {done}/{total}. Failed qa_ids: {len(set(failed_qaids))}")


[1/737] 5/3682
[2/737] 10/3682
[3/737] 15/3682
[4/737] 20/3682
[5/737] 25/3682
[6/737] 30/3682
[7/737] 35/3682
[8/737] 40/3682
[9/737] 45/3682
[10/737] 50/3682
[11/737] 55/3682
[12/737] 60/3682
[13/737] 65/3682
[14/737] 70/3682
[15/737] 75/3682
[16/737] 80/3682
[17/737] 85/3682
[18/737] 90/3682
[19/737] 95/3682
[20/737] 100/3682
[21/737] 105/3682
[22/737] 110/3682
[23/737] 115/3682
[24/737] 120/3682
[25/737] 125/3682
[26/737] 130/3682
[27/737] 135/3682
[28/737] 140/3682
[29/737] 145/3682
[30/737] 150/3682
[31/737] 155/3682
[32/737] 160/3682
[33/737] 165/3682
[34/737] 170/3682
[35/737] 175/3682
[36/737] 180/3682
[37/737] 185/3682
[38/737] 190/3682
[39/737] 195/3682
[40/737] 200/3682
[41/737] 205/3682
[42/737] 210/3682
[43/737] 215/3682
[44/737] 220/3682
[45/737] 225/3682
[46/737] 230/3682
[47/737] 235/3682
[48/737] 240/3682
[49/737] 245/3682
[50/737] 250/3682
[51/737] 255/3682
[52/737] 260/3682
[53/737] 265/3682
[54/737] 270/3682
[55/737] 275/3682
[56/737] 280/3682
[57/737] 285/3682
[58